# discover_plugins()

> Notebook generated from [`examples/extension/discover_plugins_demo.py`](../../examples/extension/discover_plugins_demo.py).

| Field | Value |
|------|-------|
| **Dataset** | Installed entry points |
| **API key** | ✅ **No API keys required** — uses stubs/mocks. |

**Expected result:** Iterates over 4 entry-point groups and reports discovered plugins.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Demonstrate plugin discovery with an in-memory entry point.

This monkeypatches the entry-point seam so the example runs without installing
a real plugin distribution. In production, ``discover_plugins()`` reads entry
points declared in installed packages' ``pyproject.toml``.

Run::

    python examples/extension/discover_plugins_demo.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

from importlib.metadata import EntryPoint

from prismal.agents.extension import PrismalStateGraphBuilder, discover_plugins
from prismal.agents.extension import plugins as plugins_mod
from prismal.agents.subgraphs.registry import SubgraphRegistry
from prismal.core.config import Settings


async def _node(state: dict) -> dict:
    return {"metadata": {"demo": True}}


def register_demo_pipeline(registry: SubgraphRegistry) -> None:
    builder = PrismalStateGraphBuilder("demo_plugin_pipeline")
    builder.add_node("n", _node)
    builder.set_entry_point("n")
    builder.add_edge("n", "__end__")
    registry.register_sync("demo_plugin_pipeline", builder.compile())


def main() -> None:
    ep = EntryPoint(
        name="demo_plugin_pipeline",
        value=f"{__name__}:register_demo_pipeline",
        group="prismal.subgraphs",
    )
    plugins_mod._entry_points = lambda group: [ep] if group == "prismal.subgraphs" else []

    registry = SubgraphRegistry()
    report = discover_plugins(
        settings=Settings(plugins_autodiscover=True),
        registry=registry,
        groups=["subgraphs"],
    )
    print(f"loaded={report.loaded_count} failed={report.failed_count}")
    print("registered subgraphs:", registry.list())


if __name__ == "__main__":
    main()

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()